# Session Analysis

Add more cells below as needed (one per analyze script).

**Before running:** set `DATA_FOLDER` to the folder containing your
session JSON files. This notebook expects `parse_sessions.py` (and any
`analyze_*.py` scripts you add cells for) to sit in the same folder as
this notebook, since it imports them as modules.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))

strategies = ["MACDTarget", "MACDTrail",  "MACDFlip"]
days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]

STRATEGY = strategies[0]  # 0-MACDTarget, 1-MACDTrail, 2-MACDFlip
DAY = days[1]            # 0-Monday / 1-Tuesday / 2-Wednesday / 3-Thursday / 4-Friday

BASE_DIR = Path(r"C:\Git\CandleStateSessionAnalysis\data")
DATA_FOLDER = BASE_DIR / STRATEGY
OUTPUT_FOLDER = DATA_FOLDER / "output" 


In [ ]:
import parse_sessions

tables = parse_sessions.parse_sessions(DATA_FOLDER)
parse_sessions.save_tables(tables, OUTPUT_FOLDER)


In [ ]:
import analyze_stats

positions, sessions, transactions = analyze_stats.load_positions_and_sessions(OUTPUT_FOLDER)
daily = analyze_stats.daily_results(positions, sessions, transactions)
trading_capital = analyze_stats.get_trading_capital(sessions)

stats = analyze_stats.compute_stats(daily, trading_capital)

print("--- Summary statistics ---")
print(analyze_stats.format_for_display(stats).to_string())

In [ ]:
import analyze_DOW

positions, sessions = analyze_DOW.load_positions_and_sessions(OUTPUT_FOLDER)
daily = analyze_DOW.daily_results(positions, sessions)

print("--- Performance by day of week ---")
dow = analyze_DOW.day_of_week_summary(daily)
display(analyze_DOW.format_for_display(dow))

streaks = analyze_DOW.longest_streaks(daily)
print(f"\nLongest Win Streak:  {streaks['LongestWinStreak']}")
print(f"Longest Loss Streak: {streaks['LongestLossStreak']}")

print(f"\n{analyze_DOW.current_streak_summary(daily)}")



In [ ]:
import analyze_day

try:
        weekday = analyze_day.normalize_weekday(DAY)
except ValueError as e:
        print(f"Error: {e}")
else:

        positions, sessions = analyze_day.load_positions_and_sessions(OUTPUT_FOLDER)
        days = analyze_day.day_results(positions, sessions, weekday)

        if days.empty:
                print(f"No {DAY} sessions found in this data.")
        else:
                print(f"--- {len(days)} {DAY}s, sorted worst to best ---")
                print(days.to_string(index=False))

                outliers = days[days["IsOutlier"]]
                if not outliers.empty:
                        print(f"\n{len(outliers)} outlier day(s) (>1 std below the {weekday} average):")
                        print(outliers[["Date", "DailyNet"]].to_string(index=False))
                else:
                        print(f"\nNo single-day outliers -- {weekday}'s result looks spread across many days, "
                        "not driven by one or two disasters.")

                print(f"\n--- Positions from the 3 worst {weekday}s ---")
                worst = analyze_day.worst_day_positions(positions, days)
                print(worst.to_string(index=False))  

In [ ]:
import  analyze_symbol

positions = analyze_symbol.load_positions(OUTPUT_FOLDER)
summary = analyze_symbol.symbol_summary(positions)

print("--- Performance by symbol ---")
print(analyze_symbol.format_for_display(summary).to_string())



In [ ]:
from datetime import datetime
import html as html_module
import re
import pandas as pd


def df_to_markdown_table(df, index_label=None, show_index=True):
    """
    Renders a DataFrame as a GitHub-flavored markdown table without
    needing the `tabulate` package (which may not be installed).
    show_index=False drops a plain positional index (e.g. sorted-but-not-
    meaningfully-indexed tables like `days` or `worst`).
    """
    df = df.reset_index(drop=not show_index)
    if index_label and show_index:
        df = df.rename(columns={df.columns[0]: index_label})
    header = "| " + " | ".join(str(c) for c in df.columns) + " |"
    sep = "| " + " | ".join("---" for _ in df.columns) + " |"
    rows = ["| " + " | ".join(str(v) for v in row) + " |" for row in df.itertuples(index=False)]
    return "\n".join([header, sep] + rows)

def makePlural(word: str, count: int) -> str:
    if count == 1:
        return word
    if word.endswith("s"):
        return f"{word}es"
    return f"{word}s"

sessions_df = pd.read_csv(OUTPUT_FOLDER / "sessions.csv")
transactions_df = pd.read_csv(OUTPUT_FOLDER / "transactions.csv")

session_dates = pd.to_datetime(sessions_df["SessionStart"], utc=True).dt.tz_convert("America/New_York")
from_date = session_dates.min().strftime("%Y-%m-%d")
thru_date = session_dates.max().strftime("%Y-%m-%d")
thru_date_compact = session_dates.max().strftime("%Y%m%d")
trading_days = session_dates.count()

trading_capital = analyze_stats.get_trading_capital(sessions_df)

targets = sessions_df["DayTarget"].dropna().unique()
if len(targets) > 1:
    print(f"WARNING: DayTarget varies across sessions {sorted(targets)} "
          f"-- using the first session's value ({targets[0]:,.0f}) for the report header.")
day_target = float(targets[0])

sections = [
    f"# Session Analysis Report\n\n"
    f"**Strategy:** {STRATEGY}  \n"
    f"**Period:** {from_date} to {thru_date} ({trading_days} days)  \n"
    f"**Trading Capital:** ${trading_capital:,.0f}  \n"
    f"**Day Target:** ${day_target:,.0f}  \n"
    f"**Generated:** {datetime.now():%Y-%m-%d %H:%M}"
]

# --- Summary statistics ---
positions_df = pd.read_csv(OUTPUT_FOLDER / "positions.csv")
daily_net = positions_df.groupby("SourceFile", as_index=False).agg(
    NetValue=("RealizedGain", "sum"),
    NumberOfPositions=("RealizedGain", "count"),
)

recent = sessions_df[["SourceFile", "SessionStart", "TargetHit"]].merge(daily_net, on="SourceFile", how="left")
recent["NetValue"] = recent["NetValue"].fillna(0)
recent["SessionStart_dt"] = pd.to_datetime(recent["SessionStart"], utc=True).dt.tz_convert("America/New_York")
recent = recent.sort_values("SessionStart_dt").tail(5)  # oldest-first within this 5-day window after ascending sort

recent_display = pd.DataFrame({
    "Date": recent["SessionStart_dt"].map(lambda dt: f"{dt.strftime('%a, %b')} {dt.day}"),
    "Positions": recent["NumberOfPositions"],
    "TargetHit": recent["TargetHit"].fillna(0).map(lambda v: "Yes" if v else "No"),
    "Net": recent["NetValue"].map(lambda v: f"-${abs(v):,.0f}" if v < 0 else f"${v:,.0f}"),
})

if "stats" in globals():
    stats_display = analyze_stats.format_for_display(stats).to_frame(name="Value")
    section = "## Summary Statistics\n\n" + df_to_markdown_table(stats_display, index_label="Metric")
else:
    section = "## Summary Statistics\n\n_Stats table not available -- run the analyze_stats cell first._"

daily_for_monthly = analyze_stats.daily_results(positions_df, sessions_df, transactions_df)
monthly = analyze_stats.monthly_results(daily_for_monthly)
monthly_display = analyze_stats.format_for_monthly_display(monthly)

# --- MONTHLY RESULTS ---
if not monthly_display.empty:
    section += "\n\n### Monthly Results\n\n" + df_to_markdown_table(monthly_display, show_index=False)

# --- LAST 5 DAYS ---
if not recent_display.empty:
    section += "\n\n### Last 5 Days\n\n" + df_to_markdown_table(recent_display, show_index=False)

sections.append(section)

# --- Day of week ---
if "dow" in globals() and "streaks" in globals():
    dow_display = analyze_DOW.format_for_display(dow)
    
    # --- Streaks ---
    streak_lines = (
        f"\n\n**Longest Win Streak:** {streaks['LongestWinStreak']} {makePlural('day', streaks['LongestWinStreak'])}  \n"
        f"**Longest Loss Streak:** {streaks['LongestLossStreak']} {makePlural('day', streaks['LongestLossStreak'])}"
    )
    streak_lines += f"  \n{analyze_DOW.current_streak_summary(daily)}"

    # if "current" in globals() and current.get("Type") is not None:
    #     verb = "winning" if current["Type"] == "Win" else "losing"
    #     streak_lines += f"  \n**Current Streak:** {current['Length']} {verb} {makePlural('day', current['Length'])}"
    
    sections.append(
        "## Performance by Day of Week\n\n"
        + df_to_markdown_table(dow_display, index_label="Weekday")
        + streak_lines
    )
else:
    sections.append("## Performance by Day of Week\n\n_Not available -- run the analyze_DOW cell first._")

# --- Single-day deep dive ---
""" if "weekday" in globals() and "days" in globals() and not days.empty:
    day_section = f"## {weekday} Deep Dive\n\n" + df_to_markdown_table(days, show_index=False)
    if "worst" in globals():
        day_section += f"\n\n### Positions from the 3 worst {weekday}s\n\n" + df_to_markdown_table(worst, show_index=False)
    sections.append(day_section)
else:
    sections.append("## Single-Day Deep Dive\n\n_Not available -- run the analyze_day cell first (with a valid DAY)._") """

# --- Symbol breakdown ---
if "summary" in globals():
    symbol_display = analyze_symbol.format_for_display(summary)
    sections.append("## Performance by Symbol\n\n" + df_to_markdown_table(symbol_display, index_label="Symbol"))
else:
    sections.append("## Performance by Symbol\n\n_Not available -- run the analyze_symbol cell first._")

report_md = "\n\n".join(sections)

# report_md_path = OUTPUT_FOLDER / f"report_{thru_date_compact}.md"
# report_md_path.write_text(report_md, encoding="utf-8")
# print(f"Wrote {report_md_path}")

# --- Also render a simple standalone HTML version ---


def markdown_table_to_html(md_table: str) -> str:
    lines = [l for l in md_table.strip().splitlines() if l.strip()]
    header_cells = [c.strip() for c in lines[0].strip("|").split("|")]
    body_rows = [[c.strip() for c in l.strip("|").split("|")] for l in lines[2:]]
    thead = "<tr>" + "".join(f"<th>{html_module.escape(c)}</th>" for c in header_cells) + "</tr>"
    tbody = "".join(
        "<tr>" + "".join(f"<td>{html_module.escape(c)}</td>" for c in row) + "</tr>"
        for row in body_rows
    )
    return f"<table>\n<thead>{thead}</thead>\n<tbody>{tbody}</tbody>\n</table>"


def section_to_html(section_md: str) -> str:
    lines = section_md.split("\n\n")
    html_parts = []
    for block in lines:
        stripped = block.strip()
        if stripped.startswith("### "):
            html_parts.append(f"<h3>{html_module.escape(stripped[4:])}</h3>")
        elif stripped.startswith("## "):
            html_parts.append(f"<h2>{html_module.escape(stripped[3:])}</h2>")
        elif stripped.startswith("# "):
            html_parts.append(f"<h1>{html_module.escape(stripped[2:])}</h1>")
        elif stripped.startswith("|"):
            html_parts.append(markdown_table_to_html(stripped))
        elif stripped.startswith("_") and stripped.endswith("_"):
            html_parts.append(f"<p><em>{html_module.escape(stripped[1:-1])}</em></p>")
        else:
            # Escape first, then convert **bold** markers and markdown
            # line breaks ("  \n", two trailing spaces) to <br>.
            escaped = html_module.escape(stripped)
            escaped = re.sub(r"\*\*(.+?)\*\*", r"<strong>\1</strong>", escaped)
            escaped = escaped.replace("  \n", "<br>\n")
            html_parts.append(f"<p>{escaped}</p>")
    return "\n".join(html_parts)


body_html = "\n".join(section_to_html(s) for s in sections)
report_html = f"""<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<title>Session Analysis Report - {html_module.escape(STRATEGY)}</title>
<style>
  body {{ font-family: -apple-system, Segoe UI, sans-serif; max-width: 650px; margin: 1.5rem auto; padding: 0 1rem; color: #1a1a1a; font-size: 11px; line-height: 1.3; }}
  table {{ border-collapse: collapse; width: auto; min-width: 320px; margin: 0.5rem 0 0.75rem; }}
  th, td {{ border: 1px solid #ddd; padding: 2px 10px; text-align: right; }}
  th:first-child, td:first-child {{ text-align: left; }}
  th {{ background: #f4f4f4; }}
  h1 {{ border-bottom: 2px solid #333; padding-bottom: 0.3rem; margin-bottom: 0.5rem; }}
  h2 {{ margin-top: 1.5rem; margin-bottom: 0.5rem; border-bottom: 1px solid #ccc; padding-bottom: 0.2rem; }}
  h3 {{ margin-top: 1rem; margin-bottom: 0.4rem; }}
  p {{ margin: 0.3rem 0; }}
</style>
</head>
<body>
{body_html}
</body>
</html>"""

report_html_path = OUTPUT_FOLDER / f"report_{thru_date_compact}.html"
report_html_path.write_text(report_html, encoding="utf-8")
print(f"Wrote {report_html_path}")